# 06 — Middleware: Intercepting Agent Behavior

**What you'll learn:**
- The three middleware layers: Agent, Function, Chat
- Building compliance logging for regulatory audit trails
- PII detection that blocks sensitive data from reaching the LLM
- Function timing for performance monitoring

---

## Why Middleware?

In banking and insurance, you can't just ship an agent and hope for the best.
Regulators require **audit trails**, security teams need **PII protection**,
and operations needs **performance metrics**.

Middleware lets you intercept agent behavior at three levels:

| Layer | Intercepts | Use Cases |
|-------|-----------|----------|
| **Agent** | The entire `agent.run()` call | Audit logging, input validation, result override |
| **Function** | Individual tool calls | Timing, argument validation, result caching |
| **Chat** | Raw LLM request/response | Token counting, prompt injection detection |

Each middleware calls `await call_next()` to continue the chain, or skips it
to block execution.

In [1]:
import os
import re
import time
from collections.abc import Awaitable, Callable
from typing import Annotated

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import Field

from agent_framework import (
    AgentContext,
    AgentMiddleware,
    AgentResponse,
    FunctionInvocationContext,
    FunctionMiddleware,
    Message,
    MiddlewareTermination,
    tool,
)
from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()

## Middleware 1: Audit Logging (Class-Based Agent Middleware)

Every interaction gets logged with a timestamp — essential for regulatory
compliance in financial services.

In [2]:
from datetime import datetime, timezone


class AuditLoggingMiddleware(AgentMiddleware):
    """Logs every agent interaction for regulatory audit trails."""

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        timestamp = datetime.now(timezone.utc).isoformat()
        last_message = context.messages[-1] if context.messages else None
        user_input = last_message.text[:100] if last_message and last_message.text else "N/A"

        print(f"[AUDIT {timestamp}] Agent={context.agent.name} | Input=\"{user_input}\"")

        await call_next()

        response_text = context.result.text[:100] if context.result and context.result.text else "N/A"
        print(f"[AUDIT {timestamp}] Agent={context.agent.name} | Output=\"{response_text}\"")


print("AuditLoggingMiddleware defined")

AuditLoggingMiddleware defined


## Middleware 2: PII Guard (Function-Based Agent Middleware)

Detects Social Security numbers, credit card numbers, and bank account
numbers in the user's input. If PII is found, it **terminates** the
request before it ever reaches the LLM.

In [3]:
# Patterns for common PII in financial services
PII_PATTERNS = {
    "SSN": r"\b\d{3}-\d{2}-\d{4}\b",
    "Credit Card": r"\b(?:\d{4}[- ]?){3}\d{4}\b",
    "Bank Account": r"\b\d{8,17}\b",  # simplified pattern
}


async def pii_guard_middleware(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Blocks requests containing PII before they reach the model."""
    last_message = context.messages[-1] if context.messages else None
    if last_message and last_message.text:
        for pii_type, pattern in PII_PATTERNS.items():
            if re.search(pattern, last_message.text):
                print(f"[PII GUARD] ⚠ Detected {pii_type} in input — BLOCKING request")
                context.result = AgentResponse(
                    messages=[
                        Message(
                            role="assistant",
                            text=(
                                f"I detected what appears to be a {pii_type} in your message. "
                                "For your security, please don't share sensitive personal "
                                "information in chat. Contact us through our secure portal instead."
                            ),
                        )
                    ]
                )
                raise MiddlewareTermination(result=context.result)

    print("[PII GUARD] ✓ No PII detected")
    await call_next()


print("pii_guard_middleware defined")

pii_guard_middleware defined


## Middleware 3: Function Timing (Class-Based Function Middleware)

Logs how long each tool call takes — useful for monitoring slow API
calls or database queries in production.

In [4]:
class FunctionTimingMiddleware(FunctionMiddleware):
    """Logs execution time for every function/tool call."""

    async def process(
        self,
        context: FunctionInvocationContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        fn_name = context.function.name
        print(f"[TIMING] Calling {fn_name}...")

        start = time.perf_counter()
        await call_next()
        elapsed = time.perf_counter() - start

        print(f"[TIMING] {fn_name} completed in {elapsed:.4f}s")


print("FunctionTimingMiddleware defined")

FunctionTimingMiddleware defined


## Putting It All Together

Create a banking agent with all three middleware layers.

In [5]:
@tool(approval_mode="never_require")
def get_account_balance(
    account_type: Annotated[str, Field(description="Account type: 'checking' or 'savings'")],
) -> str:
    """Look up the current balance for a customer's account."""
    balances = {"checking": 3_842.51, "savings": 15_290.00}
    balance = balances.get(account_type)
    if balance is None:
        return f"Unknown account type: {account_type}"
    return f"{account_type.title()} account balance: ${balance:,.2f}"


@tool(approval_mode="never_require")
def get_recent_transactions(
    account_type: Annotated[str, Field(description="Account type: 'checking' or 'savings'")],
    count: Annotated[int, Field(description="Number of recent transactions to return")] = 3,
) -> str:
    """Get recent transactions for a customer's account."""
    transactions = {
        "checking": [
            "Mar 20 — Grocery Store — -$67.43",
            "Mar 19 — Direct Deposit — +$2,450.00",
            "Mar 18 — Electric Bill — -$142.00",
            "Mar 15 — Restaurant — -$38.90",
        ],
        "savings": [
            "Mar 01 — Interest Credit — +$12.74",
            "Feb 15 — Transfer from Checking — +$500.00",
            "Feb 01 — Interest Credit — +$11.98",
        ],
    }
    txns = transactions.get(account_type, [])
    return "\n".join(txns[:count]) or "No transactions found."


print("Banking tools defined")

Banking tools defined


In [6]:
banking_agent = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
).as_agent(
    name="BankingAgent",
    instructions=(
        "You are a helpful banking assistant at First National Bank. "
        "Use the available tools to look up account balances and transactions. "
        "Be professional and concise."
    ),
    tools=[get_account_balance, get_recent_transactions],
    middleware=[
        AuditLoggingMiddleware(),
        pii_guard_middleware,
        FunctionTimingMiddleware(),
    ],
)

print("Banking agent created with 3 middleware layers")

Banking agent created with 3 middleware layers


## Demo 1: Normal Query (All Middleware Passes)

Watch the audit log + timing middleware fire as the agent processes the request.

In [7]:
result = await banking_agent.run("What's my checking account balance and last 3 transactions?")
print(f"\nAgent: {result.text}")

[AUDIT 2026-03-25T01:45:28.376191+00:00] Agent=BankingAgent | Input="What's my checking account balance and last 3 transactions?"
[PII GUARD] ✓ No PII detected
[TIMING] Calling get_account_balance...
[TIMING] get_account_balance completed in 0.0001s
[TIMING] Calling get_recent_transactions...
[TIMING] get_recent_transactions completed in 0.0000s
[AUDIT 2026-03-25T01:45:28.376191+00:00] Agent=BankingAgent | Output="Your checking account balance is $3,842.51. Here are your last three transactions:

1. **Mar 20**: G"

Agent: Your checking account balance is $3,842.51. Here are your last three transactions:

1. **Mar 20**: Grocery Store — **-$67.43**  
2. **Mar 19**: Direct Deposit — **+$2,450.00**  
3. **Mar 18**: Electric Bill — **-$142.00**  


## Demo 2: PII Detected (Request Blocked)

The PII guard middleware will catch the SSN and terminate the request
before it reaches the LLM.

In [8]:
result = await banking_agent.run(
    "My SSN is 123-45-6789, can you look up my account?"
)
print(f"\nAgent: {result.text}")

[AUDIT 2026-03-25T01:45:34.279048+00:00] Agent=BankingAgent | Input="My SSN is 123-45-6789, can you look up my account?"
[PII GUARD] ⚠ Detected SSN in input — BLOCKING request

Agent: I detected what appears to be a SSN in your message. For your security, please don't share sensitive personal information in chat. Contact us through our secure portal instead.


## Demo 3: Run-Level Middleware

You can add **per-request middleware** that only applies to a specific run.
Agent-level middleware still runs — the run-level middleware is added inside.

In [9]:
async def custom_disclaimer_middleware(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Appends a legal disclaimer to the agent's response."""
    await call_next()

    if context.result and context.result.text:
        disclaimer = "\n\n_Disclaimer: This is for informational purposes only and does not constitute financial advice._"
        # Modify the result by creating a new response with the disclaimer
        original_text = context.result.text
        context.result = AgentResponse(
            messages=[Message(role="assistant", text=original_text + disclaimer)]
        )
        print("[DISCLAIMER] Appended legal disclaimer")


# This run gets the disclaimer middleware ON TOP of the agent-level middleware
result = await banking_agent.run(
    "What's my savings account balance?",
    middleware=[custom_disclaimer_middleware],
)
print(f"\nAgent: {result.text}")

[AUDIT 2026-03-25T01:45:34.299164+00:00] Agent=BankingAgent | Input="What's my savings account balance?"
[PII GUARD] ✓ No PII detected
[TIMING] Calling get_account_balance...
[TIMING] get_account_balance completed in 0.0000s
[DISCLAIMER] Appended legal disclaimer
[AUDIT 2026-03-25T01:45:34.299164+00:00] Agent=BankingAgent | Output="Your savings account balance is $15,290.00.

_Disclaimer: This is for informational purposes only an"

Agent: Your savings account balance is $15,290.00.

_Disclaimer: This is for informational purposes only and does not constitute financial advice._


In [10]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- **Agent middleware** wraps the entire `agent.run()` — great for logging and input validation
- **Function middleware** wraps individual tool calls — great for timing and caching
- **Chat middleware** wraps the raw LLM call — great for token counting
- `MiddlewareTermination` blocks execution and returns a custom response
- Middleware can be **agent-level** (all runs) or **run-level** (per-request)
- Execution order: Agent-level → Run-level → Agent execution

**Next up:** [07 — Workflows](./07-workflows-intro.ipynb) — orchestrate
multi-step processes like loan application pipelines.